# W04｜数据库 Schema / ORM / State｜教学版

**建议学习时间：5h**

这本 Notebook 的定位是“第二教材”，不是代码仓库。  
正确使用方式：

1. 先看课程视频；
2. 回到 Notebook 看“是什么 / 为什么 / 什么时候用”；
3. 带着“你应该观察什么”再运行代码；
4. 先自己做题，再展开参考答案；
5. 最后把同一机制接入 ChatBI。

## Excel 任务表 1:1 对照

| # | 模块 | Excel任务 | 输出/验收 | Notebook处理方式 |
|---:|---|---|---|---|
| 1 | Backend | 按下方课程映射完成 SQL/SQLModel/PostgreSQL/LangGraph Core | 完成 Ch7/8/10 + Alembic/1:N + LangGraph Core | 知识点 + Checkpoint/参考答案 |
| 2 | Agent | 学 Agent State vs Business State | 知道哪些状态要持久化 | 知识点 + Checkpoint/参考答案 |
| 3 | Lab | 自己先画 conversation/message/job ER 图 | 再让 AI review，不让 AI 代画 | 明确作业 + 参考答案/参考实现 |
| 4 | ChatBI | 保存会话、消息和 query job | 重启服务后仍能读取历史 | 明确作业 + 参考答案/参考实现 |
| 5 | 复盘 | 记录 3 个 Schema 决策 | 能解释为什么这样拆表 | 明确作业 + 参考答案/参考实现 |

> **规则**：Task 1-2 是学习任务，用 Checkpoint/面试题验收；Task 3-5 是作业任务，下面给明确参考答案/参考实现。设计题没有唯一答案时给参考图/评分标准。

## Excel 作业参考答案 / 验收标准

### Task 3｜ER 参考图
```text
Conversation 1 ─── N Message
      |
      └─────── N QueryJob
```
`QueryJob` 独立保存 status/version/retry 生命周期。

### Task 4｜持久化验收
不要用 `:memory:` 做最终验证。写入 PostgreSQL/持久 SQLite，关闭并重新启动应用后仍能查询 conversation/message/job。

### Task 5｜三个 Schema 决策参考
1. Conversation/Message 分表；
2. QueryJob 与 Message 分表；
3. 常用查询字段建索引，但不“每列都建索引”。

> 建议：先完成作业，再对照参考答案。不要直接照抄。

## 本周学习目标

        - 理解 SQL/ORM 基础
- 自己设计 conversation/message/job schema
- 区分 Agent State 和 Business State
- 理解 Migration

        ## 本周前置检查

        - 完成 W01-W03；知道 class/type hint

        ## 本周必须记住的英文术语

        - `primary key`：主键
- `foreign key`：外键
- `session`：会话
- `migration`：迁移
- `checkpoint`：检查点
- `source of truth`：事实源

        > 可以听中文翻译，但这些英文术语一定要保留，因为代码、日志、文档和面试都会使用英文。

## 1. 数据库 Schema 不是“把所有字段塞进去”

    **是什么（What）**  
    Schema 描述实体、字段、主外键、关系和约束。

    **为什么需要（Why）**  
    好的 schema 让业务状态可持久化、可查询，并在数据库层保护重要 invariant。

    **什么时候用（When）**  
    Conversation / Message / QueryJob 等长期事实。

    **常见误区 / 容易踩坑**  
    不要把 transient agent state 全塞进业务表；也不要把所有东西都做成一个 JSON。


**中英术语**：`primary key` = 主键；`foreign key` = 外键；`relationship` = 关系；`constraint` = 约束；`migration` = 迁移

### 代码实验：参数化 SQL 防止字符串拼接风险

**这段代码想让你看什么？**

观察 `?` 参数不是把用户输入直接拼进 SQL 字符串。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE job(id INTEGER PRIMARY KEY, status TEXT)")
cur.execute("INSERT INTO job(status) VALUES (?)", ("RUNNING",))
status = "RUNNING"
print(cur.execute("SELECT * FROM job WHERE status = ?", (status,)).fetchall())

## 2. Engine / Session / ORM Model / API Schema

    **是什么（What）**  
    Engine 管理数据库连接能力；Session 表示一段数据访问工作上下文；ORM Model 映射数据库；API Schema 定义对外 contract。

    **为什么需要（Why）**  
    如果这些概念混在一起，很容易出现 session 泄漏、返回敏感字段、难以测试等问题。

    **什么时候用（When）**  
    FastAPI request-scoped DB session、Repository。

    **常见误区 / 容易踩坑**  
    ORM model 不等于 API response；Session 也不应该全局共享给所有请求。


**中英术语**：`engine` = 数据库引擎/连接入口；`session` = 会话/工作单元；`ORM` = 对象关系映射；`request-scoped` = 请求级

### 代码实验：用 SQLAlchemy 建立 Conversation 1:N Message

**这段代码想让你看什么？**

观察 ForeignKey 和 relationship 分别解决数据库约束与 Python 对象导航。

> 不要先看代码细节。先带着上面的观察目标运行，再回头逐行读。

In [ ]:
from sqlalchemy import create_engine, String, ForeignKey, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

class Base(DeclarativeBase):
    pass

class Conversation(Base):
    __tablename__ = "conversation"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(100))
    messages: Mapped[list["Message"]] = relationship(
        back_populates="conversation", cascade="all, delete-orphan"
    )

class Message(Base):
    __tablename__ = "message"
    id: Mapped[int] = mapped_column(primary_key=True)
    conversation_id: Mapped[int] = mapped_column(ForeignKey("conversation.id"))
    role: Mapped[str] = mapped_column(String(20))
    content: Mapped[str] = mapped_column(String(500))
    conversation: Mapped[Conversation] = relationship(back_populates="messages")

engine = create_engine("sqlite+pysqlite:///:memory:")
Base.metadata.create_all(engine)

with Session(engine) as session:
    c = Conversation(title="Demo")
    c.messages.append(Message(role="user", content="Show monthly trend"))
    session.add(c)
    session.commit()

    result = session.scalars(select(Conversation)).first()
    print(result.title, result.messages[0].content)

## 3. Agent State vs Business State

    **是什么（What）**  
    Agent State 是一次 workflow 执行中的上下文；Business State 是需要长期保持一致的业务事实。

    **为什么需要（Why）**  
    把两者混为一谈会导致：checkpoint 恢复了，但业务状态已经不一致；或数据库保存大量无意义临时状态。

    **什么时候用（When）**  
    LangGraph state / checkpoint vs query_job.status / version。

    **常见误区 / 容易踩坑**  
    Checkpoint 不是业务数据库；business DB 也不应该承担所有 agent scratchpad。


**中英术语**：`agent state` = 智能体运行状态；`business state` = 业务状态；`checkpoint` = 检查点/执行恢复点；`source of truth` = 事实源

In [ ]:
business_state = {"job_id": "J001", "status": "RUNNING", "version": 3}
agent_state = {
    "messages": ["user: show trend"],
    "current_node": "generate_plan",
    "tool_observations": []
}

print("business state:", business_state)
print("agent state:", agent_state)

### 练习：自己设计 QueryJob Schema

至少包含 `id / conversation_id / status / version / created_at`。想一想哪些字段需要 index。

<details>
<summary><b>提示</b></summary>

经常按 conversation_id 查历史 job，按 status 查待处理任务，这些是 index 候选。

</details>

In [ ]:
# TODO: 先用 dataclass 或 Pydantic 写出你自己的 QueryJob，不要让 AI 直接生成。

<details>
    <summary><b>参考答案</b></summary>

    ```python
from dataclasses import dataclass
from datetime import datetime

@dataclass
class QueryJob:
    id: str
    conversation_id: str
    status: str
    version: int
    created_at: datetime
```

    </details>

### Checkpoint

1. 为什么 ORM Model 和 API Schema 最好分开？
2. 为什么需要 Migration？
3. Checkpoint 为什么不能代替 Business DB？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 数据库字段可能包含内部字段/关系，对外 API contract 有不同生命周期和安全要求。

**2.** Schema 会演进；Migration 让数据库结构变更可版本化、可审核、可部署/回滚。

**3.** Checkpoint 解决工作流恢复；业务事实的一致性、约束和查询仍需要业务数据库。

    </details>

## ChatBI 本周作业

        先自己画 ER 图，再实现：
`conversation 1:N message`
`conversation 1:N query_job`

项目里开始持久化 conversation / message / job。

### 本周面试 / 理解检查

1. 500 个请求是不是应该创建 500 个 DB connection？
2. Session 的生命周期为什么重要？
3. Agent state 与 business state 最大区别？

<details>
    <summary><b>参考答案（建议先自己回答）</b></summary>

    **1.** 不是。通常通过 connection pool 复用有限连接；数据库连接本身昂贵且数据库容量有限。

**2.** Session 太长会持有连接/事务和缓存状态，容易造成连接耗尽与长事务。

**3.** Agent state 主要服务执行流程；business state 是长期业务事实和系统 source of truth。

    </details>

## 本周结束前的自检

只有下面 4 项都能做到，才算真正完成：

- [ ] 我能不用课程原话，用自己的话解释核心概念；
- [ ] 我知道这周的代码实验到底在证明什么；
- [ ] 我至少独立完成了一道代码题；
- [ ] 我能说出一个“生产环境里如果做错会发生什么”的例子。